# An algorithm to find cospectral graphs

Switching algorithm that works for any graph and any switching method. Works particularly well on symmetric graphs.

Faster if *bliss* is installed.

In [1]:
# given an nxn matrix Q:
# subsets of {0,...,n-1} whose characteristic vectors are the 01-vectors x such that Q^T*x is a 01-vector again
def blocks(Q):
    n = Q.nrows()
    L = []
    for x in GF(2)^n:
        x = vector(QQ(i) for i in x)
        if all(i==0 or i==1 for i in Q.T*x):
            L.append({i for i in range(n) if x[i] == 1})
    return L

## Iterating through all switching sets

In [2]:
def allSwitchingSets(G: Graph, Q: matrix, switching_subgraph=False):
    
    n = Q.nrows() # size of the switching set
    
    # defining big graph H that is the disjoint union of G, Gamma (pointset; switching_subgraph) and B (blockset)
    if switching_subgraph:
        assert set(switching_subgraph.vertices()) == set(range(n))
        H = G.disjoint_union(switching_subgraph)
    else:
        H = G.disjoint_union(Graph(n))
    vertices = {(0,v) for v in G}
    points = {(1,p) for p in range(n)}
    blockset = []
    for b, B in enumerate(blocks(Q)):
        blockset.append(b)
        H.add_vertex(b)
        H.add_edges(((1, p), b) for p in B)
    # recursively adding vertices to the switching set, see function below
    yield from chooseRecursively(H, [], vertices, [], points, blockset, {b:[] for b in blockset}, {v:[] for v in vertices}, {p:vertices for p in points}, switching_subgraph)

In [3]:
def chooseRecursively(H, chosenVertices, availableVertices, chosenPoints, availablePoints, blockset, blockpatterns, vertexpatterns, candidates, switching_subgraph):
    # chosenVertices and chosenPoints: vertices of G resp. Gamma that are already matched (in that order)
    # availableVertices and availablePoints: remaining vertices of G resp. Gamma
    # blockpatterns and vertexpatterns: lists of booleans indicating adjacency with elements of the pointset
    # candidates: keeps track of which vertices can still be matched to each point
    # switching_subgraph: induced subgraph on pointset. If False: do not check if adjacency in G matches adjacency in Gamma

    # heuristic: does the last chosen vertex have unique degree?
    if chosenVertices:
        degrees = [len(set(chosenVertices).intersection(H.neighbors(v))) for v in chosenVertices]
        unique_degrees = [d for d in degrees if degrees.count(d) == 1]
        if unique_degrees:
            # heuristic works
            if degrees[-1] != min(unique_degrees):
                # not canonical
                return

    aut = H.automorphism_group(partition=[chosenVertices, availableVertices, chosenPoints, availablePoints, blockset], algorithm='bliss')
    
    
    # union find: choose a match between G and Gamma up to symmetry
    chosenMatches = [(chosenVertices[i],chosenPoints[i]) for i in range(len(chosenVertices))]
    availableMatches = [(v,p) for v in availableVertices for p in availablePoints]
    chosenMatches_orbits = DisjointSet(chosenMatches)
    availableMatches_orbits = DisjointSet(availableMatches)
    for f in aut.gens():
        for (v,p) in chosenMatches:
            chosenMatches_orbits.union((v,p), (f(v),f(p)))
        for (v,p) in availableMatches:
            availableMatches_orbits.union((v,p), (f(v),f(p)))

    # is the last chosen vertex canonical?
    # only if heuristic did not work
    if chosenVertices and not unique_degrees:
        # applying canonical labeling of entire graph
        _,Hcanonical = H.canonical_label(partition=[chosenVertices, availableVertices, chosenPoints, availablePoints, blockset], certificate=True , algorithm='bliss')
        canonical_vertex = min(chosenVertices, key=lambda v: Hcanonical[v])
        canonical_match = (canonical_vertex,chosenPoints[chosenVertices.index(canonical_vertex)])
        # calculate if the last chosen match belongs to the orbit of the canonical matching edge
        if chosenMatches_orbits.find((chosenVertices[-1],chosenPoints[-1])) != chosenMatches_orbits.find(canonical_match):
            # not canonical
            return
    
    # switching set found
    if len(availablePoints) == 0:
        yield [chosenVertices[chosenPoints.index((1, i))][1] for i in range(len(chosenPoints))] # ordered switching set
    
    # add match
    for (v,p) in availableMatches:
        if switching_subgraph and any(H.has_edge(v, chosenVertices[i]) != H.has_edge(p, chosenPoints[i]) for i in range(len(chosenVertices))):
            continue
        if availableMatches_orbits.find((v,p)) != (v,p):
            continue
        chosenVertices.append(v)
        availableVertices.remove(v)
        chosenPoints.append(p)
        availablePoints.remove(p)
        H.add_edge(v,p)
        # update blockpatterns
        
        for b in blockset:
            blockpatterns[b].append(H.has_edge(p, b))
        block_pattern_set = {tuple(bp) for bp in blockpatterns.values()}
        # update vertexpatterns
        countmustadd = 0
        for w in availableVertices:
            vertexpatterns[w].append(H.has_edge(v, w))
            if tuple(vertexpatterns[w]) not in block_pattern_set:
                countmustadd += 1
        if countmustadd <= len(availablePoints):
            if switching_subgraph:
                newcandidates = {}
                for q in availablePoints:
                    if q in H.neighbors(p):
                        newcandidates[q] = candidates[q].intersection(H.neighbors(v))
                    else:
                        newcandidates[q] = {w for w in candidates[q] if not H.has_edge(v,w)}
                if all(newcandidates.values()):
                    yield from chooseRecursively(H, chosenVertices, availableVertices, chosenPoints, availablePoints, blockset, blockpatterns, vertexpatterns, newcandidates, switching_subgraph)
            else:
                yield from chooseRecursively(H, chosenVertices, availableVertices, chosenPoints, availablePoints, blockset, blockpatterns, vertexpatterns, candidates, switching_subgraph)
        # undo blockpatterns
        for b in blockset:
            blockpatterns[b].pop()
        # undo vertexpatterns
        for w in availableVertices:
            vertexpatterns[w].pop()
        chosenVertices.pop()
        availableVertices.add(v)
        chosenPoints.pop()
        availablePoints.add(p)
        H.delete_edge(v,p)

## Particular methods

### Switching matrices

In [4]:
switching_matrix = {}

def O(n): return zero_matrix(n)
def I(n): return identity_matrix(n)
def J(n): return matrix([[1]*n]*n)
def Y(n): return n*I(n)-J(n)

# 4
switching_matrix["GM4"] = GM4 = 1/2*(J(4)-2*I(4))

# 6
switching_matrix["GM6"] = 1/3*(J(6)-3*I(6))
switching_matrix["WQH6"] = 1/3*block_matrix([[3*I(3)-J(3),J(3)],[J(3),3*I(3)-J(3)]])
switching_matrix["AH6"] = AH6 = 1/2*block_matrix([[J(2),O(2),Y(2)],[Y(2),J(2),O(2)],[O(2),Y(2),J(2)]])
switching_matrix["IS6"] = 1/5*matrix([[2,3,3,-1,-1,-1],[3,2,-3,1,1,1],[3,-3,2,1,1,1],[-1,1,1,-2,3,3],[-1,1,1,3,-2,3],[-1,1,1,3,3,-2]])

# 7
switching_matrix["Fano7"] = 1/2*matrix.circulant([-1,1,1,0,1,0,0])

# 8
switching_matrix["GM8"] = 1/4*(J(8)-4*I(8))
switching_matrix["GM44"] = GM44 = block_diagonal_matrix([GM4,GM4])
switching_matrix["WQH8"] = 1/4*block_matrix([[4*I(4)-J(4),J(4)],[J(4),4*I(4)-J(4)]])
switching_matrix["IS8level3"] = 1/3*matrix([[2,1,0,0,1,-1,-1,1],[1,2,0,0,-1,1,1,-1],[0,0,2,1,-1,1,-1,1],[0,0,1,2,1,-1,1,-1],[1,-1,-1,1,1,2,0,0],[-1,1,1,-1,2,1,0,0],[-1,1,-1,1,0,0,1,2],[1,-1,1,-1,0,0,2,1]])
switching_matrix["IS8level5"] = 1/5*matrix.circulant([3,1,2,-1,-2,1,2,-1])

# 9
switching_matrix["ABS9"] = ABS9 = 1/3*block_matrix([[J(3),O(3),Y(3)],[Y(3),J(3),O(3)],[O(3),Y(3),J(3)]])
switching_matrix["ABS9m"] = ABS9m = 1/3*block_matrix([[J(3),O(3),-Y(3)],[-Y(3),J(3),O(3)],[O(3),-Y(3),J(3)]])
A = matrix.circulant([2,1,0])
B = matrix.circulant([0,-1,1])
switching_matrix["IS9"] = IS9 = 1/3*block_matrix([[A,B,B],[B,A,B],[B,B,A]])

# 10
switching_matrix["GM10"] = 1/5*(J(10)-5*I(10))
switching_matrix["WQH10"] = 1/5*block_matrix([[5*I(5)-J(5),J(5)],[J(5),5*I(5)-J(5)]])
switching_matrix["AH10"] = 1/2*block_matrix([[J(2),O(2),O(2),O(2),Y(2)],[Y(2),J(2),O(2),O(2),O(2)],[O(2),Y(2),J(2),O(2),O(2)],[O(2),O(2),Y(2),J(2),O(2)],[O(2),O(2),O(2),Y(2),J(2)]])

# 13
switching_matrix["SingerPG23"] = 1/3*matrix.circulant([1,1,-1,-1,0,0,1,1,0,1,0,1,-1])

### Irreducible switching subgraphs

In [5]:
switching_subgraphs = {}

# 4
switching_subgraphs["GM4"] = [G for G in graphs(4) if G.is_regular()]

# 6
switching_subgraphs["GM6"] = [Graph(A) for A in [O(6), matrix([(0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 1, 0), (0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0), (0, 1, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1), (0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0), (1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1), (1, 1, 0, 1, 1, 1), (1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 0)])]]
switching_subgraphs["WQH6"] = [Graph(A) for A in [O(6), matrix([(0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1), (0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1), (0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0), (0, 0, 1, 0, 0, 1), (0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0), (0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1), (1, 0, 0, 0, 0, 1), (0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0), (0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0), (0, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0)]),matrix([(0, 1, 1, 0, 0, 0), (1, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 0), (0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 1), (0, 0, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 1), (1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1), (1, 0, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1), (1, 1, 1, 1, 1, 0)]),matrix([(0, 1, 1, 0, 1, 0), (1, 0, 1, 0, 0, 1), (1, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 1), (1, 0, 0, 1, 0, 1), (0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 0)]),matrix([(0, 1, 1, 0, 1, 1), (1, 0, 1, 1, 0, 1), (1, 1, 0, 1, 1, 0), (0, 1, 1, 0, 1, 1), (1, 0, 1, 1, 0, 1), (1, 1, 0, 1, 1, 0)]),matrix([(0, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1), (1, 1, 0, 1, 1, 1), (1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 0)])]]
switching_subgraphs["AH6"] = [Graph(A) for A in [matrix([(0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0), (1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 0, 1), (0, 1, 0, 1, 0, 0), (0, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0)])]]
switching_subgraphs["IS6"] = [Graph(A) for A in [O(6),matrix([(0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0), (0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0), (1, 0, 1, 0, 0, 0), (1, 0, 1, 0, 0, 0)]),matrix([(0, 1, 1, 0, 0, 1), (1, 0, 0, 1, 1, 0), (1, 0, 0, 1, 1, 0), (0, 1, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1), (1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 1), (0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0), (0, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0), (0, 1, 0, 0, 1, 1), (0, 1, 0, 1, 0, 1), (0, 1, 0, 1, 1, 0)]),matrix([(0, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1), (1, 1, 0, 1, 1, 1), (1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 0)])]]

# 7
switching_subgraphs["Fano7"] = [Graph(A) for A in [matrix([(0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0), (1, 0, 0, 1, 1, 1, 1), (0, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 1), (0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 1, 1, 0, 0), (1, 0, 1, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0), (1, 0, 0, 1, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1), (1, 1, 0, 0, 0, 1, 1), (1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0)])]]

# 8
switching_subgraphs["GM8"] = [Graph(A) for A in [O(8),matrix([(0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 0), (0, 0, 1, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 1, 0, 0, 1, 0, 0), (1, 1, 0, 0, 1, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 1, 0, 0, 1, 0, 0), (0, 1, 1, 0, 1, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (0, 1, 1, 0, 0, 1, 0, 0), (1, 0, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 0), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 0), (1, 0, 1, 1, 0, 0, 1, 0), (1, 1, 1, 0, 0, 0, 0, 1), (0, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (1, 1, 1, 0, 0, 1, 1, 0), (1, 1, 1, 0, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 1, 0), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 1), (0, 0, 1, 1, 1, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 0), (1, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1, 1, 1), (1, 1, 0, 1, 1, 1, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)])]]
switching_subgraphs["GM44"] = [Graph(A) for A in [matrix([(0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 0), (1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 1, 0, 1, 0, 1, 1), (0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 0, 1, 0, 0), (1, 1, 1, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0)]),matrix([(0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 0), (1, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 1, 0, 1, 1), (0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 1), (0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 1), (1, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (1, 1, 0, 1, 1, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (1, 0, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (1, 0, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 1, 0, 1, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (1, 0, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (0, 1, 0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (0, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 0, 1, 0), (0, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 0), (1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 0), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)])]]
switching_subgraphs["WQH8"] = [Graph(A) for A in [O(8),matrix([(0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 0, 1), (0, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 1), (0, 1, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 1, 1, 0, 0), (1, 0, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 0, 0, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 0, 1), (0, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 1), (1, 0, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 1, 0, 1), (1, 0, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 1, 0), (1, 1, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 1, 1, 0, 0), (1, 0, 1, 0, 0, 0, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 1, 0, 1), (0, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 1), (0, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 0, 1, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 0, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 1, 0, 0, 1, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0), (1, 0, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1), (1, 0, 1, 1, 0, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 1, 0), (1, 0, 1, 0, 0, 1, 0, 1), (1, 0, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (0, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 1, 0, 1, 0), (1, 0, 1, 0, 1, 0, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 1, 0, 1, 0, 0, 0), (1, 0, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0, 0, 1), (1, 0, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (1, 0, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 1), (0, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 1, 0, 1, 0, 0, 1), (0, 1, 1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 1, 1), (1, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 1, 1, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 1), (0, 1, 0, 1, 1, 1, 0, 0), (1, 0, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0), (1, 1, 1, 0, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 1, 1, 0, 1, 1, 1, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 1, 1, 0, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 0, 1, 0, 1, 0), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 1, 1, 0, 0), (0, 0, 1, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 1, 1, 0, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 1), (0, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 1, 1, 0, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0), (1, 1, 1, 0, 0, 1, 0, 1), (0, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 0, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 1, 1, 0, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 0, 1, 0, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 0, 0, 0), (1, 0, 0, 1, 0, 1, 0, 0), (1, 1, 1, 0, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0, 1, 1), (0, 0, 1, 0, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (1, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 0), (1, 1, 0, 0, 1, 1, 1, 1), (1, 1, 0, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (0, 1, 1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 1, 0, 1), (1, 0, 0, 1, 1, 0, 1, 0), (1, 1, 0, 0, 0, 1, 0, 1), (0, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 1, 0, 0, 0, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 0, 1, 1, 1), (0, 1, 1, 0, 1, 1, 0, 0), (0, 1, 0, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 1, 1, 1, 1, 1), (0, 1, 1, 0, 1, 1, 1, 1), (0, 0, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 1, 0, 1, 0, 0, 1, 0), (1, 0, 1, 0, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (1, 1, 0, 0, 1, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 1, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 0, 1, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 1, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 0), (0, 0, 1, 1, 0, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 0), (1, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 0, 1, 1, 0, 0), (0, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 1, 1, 1, 0, 0, 1), (1, 1, 0, 0, 0, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 0, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 0), (0, 0, 1, 1, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 1, 1, 1, 1, 1), (0, 1, 1, 0, 1, 1, 1, 1), (1, 0, 1, 1, 0, 0, 1, 0), (0, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 1), (1, 1, 1, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 1, 0), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 1, 1, 1, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 1, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 0), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 0, 0, 1, 1), (1, 1, 1, 0, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 1, 1, 0), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 0, 0), (0, 0, 1, 1, 0, 1, 1, 1), (1, 0, 0, 1, 1, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 1), (0, 0, 1, 1, 1, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 0), (1, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 1, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1, 1, 1), (1, 1, 0, 1, 1, 1, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)])]]
switching_subgraphs["IS8level3"] = [Graph(A) for A in [matrix([(0, 1, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 0), (1, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 0, 1, 0), (1, 1, 0, 1, 1, 0, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 1, 1, 1, 1, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0)]),matrix([(0, 1, 1, 1, 0, 1, 0, 0), (1, 0, 1, 1, 1, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0, 1), (0, 1, 0, 1, 0, 1, 0, 0), (1, 0, 1, 0, 1, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 1, 0, 1, 0, 1, 1, 0), (1, 1, 1, 0, 1, 0, 0, 1), (0, 1, 0, 1, 0, 1, 0, 0), (1, 0, 1, 0, 1, 0, 0, 0), (0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 1, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 1), (0, 0, 1, 1, 1, 0, 1, 0), (1, 1, 0, 0, 0, 1, 1, 0), (1, 1, 0, 0, 1, 0, 0, 1), (0, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 0, 0, 0, 1, 1), (0, 1, 1, 0, 1, 1, 0, 0), (1, 0, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 1, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 1, 0, 1, 0, 1, 1, 0), (1, 1, 1, 0, 1, 0, 0, 1), (0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 1), (0, 1, 1, 0, 1, 1, 0, 1), (1, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)])]]
switching_subgraphs["IS8level5"] = [Graph(A) for A in [O(8),matrix([(0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 0, 1, 0), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 1, 0, 1, 0), (1, 1, 0, 1, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 0, 1, 0), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 1, 0, 1, 0), (1, 1, 0, 1, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 1, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 1, 0, 1, 0, 0), (1, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 1, 0, 0, 0, 1, 1), (1, 1, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 0, 1, 1, 0, 1, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 1, 1, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 1, 1, 0), (1, 1, 1, 0, 0, 1, 1, 0), (0, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 1), (0, 1, 1, 0, 0, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 0), (1, 0, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0), (0, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 1, 0, 0, 0, 1, 1), (1, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 0), (0, 0, 0, 1, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 0), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 0, 1, 1, 0), (1, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 1, 0, 1), (1, 1, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 1, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 1, 0, 0, 0, 0), (0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 1, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 0, 1, 1), (1, 1, 0, 1, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 1), (1, 0, 0, 0, 0, 1, 0, 0), (0, 1, 1, 0, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 0), (1, 1, 1, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 1, 0, 0, 0, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (1, 1, 1, 0, 1, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 1, 1, 0, 0, 1, 0, 0), (1, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 1, 1, 0), (0, 1, 0, 0, 0, 1, 1, 0), (1, 0, 0, 1, 1, 0, 1, 1), (0, 1, 1, 1, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 0), (1, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 1, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0, 1), (1, 0, 1, 1, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 1, 0), (1, 1, 1, 0, 1, 1, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (1, 0, 1, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 0, 0, 0), (1, 1, 0, 0, 0, 0, 1, 0), (0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 1, 1, 0, 1, 0), (0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 0, 1, 0), (0, 1, 1, 0, 0, 1, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 1, 1, 1, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 0, 1, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 1, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 1, 0), (0, 1, 1, 0, 0, 0, 0, 1), (0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 0), (0, 1, 1, 0, 0, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 1, 0, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 1, 0, 0), (1, 0, 1, 0, 0, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 1, 0, 1, 0, 0, 0, 0), (1, 0, 1, 0, 0, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 1, 0, 1, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1, 1, 0), (0, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 0, 0, 1), (1, 0, 1, 0, 0, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 1, 0, 1, 0, 0), (0, 1, 0, 1, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 0), (1, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1), (1, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 0, 1, 1, 0), (1, 1, 0, 0, 0, 1, 1, 0), (1, 0, 1, 1, 1, 0, 0, 0), (0, 0, 1, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 1, 0), (1, 0, 1, 0, 0, 1, 0, 1), (1, 1, 0, 1, 0, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 1, 0, 1, 0, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 0), (0, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 0, 0, 1), (1, 0, 1, 0, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1), (0, 1, 1, 0, 1, 0, 1, 1), (0, 0, 1, 1, 1, 1, 0, 1), (1, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 0, 1, 1, 0), (0, 1, 1, 0, 0, 1, 1, 1), (0, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 1, 1, 0, 0, 1, 0, 0), (1, 0, 1, 1, 1, 0, 0, 1), (1, 1, 0, 1, 0, 0, 1, 0), (0, 1, 1, 0, 0, 1, 0, 0), (0, 1, 0, 0, 0, 1, 1, 0), (1, 0, 0, 1, 1, 0, 1, 1), (0, 0, 1, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 0, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 0, 0), (1, 0, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 0), (0, 1, 1, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 1, 1, 1), (0, 1, 1, 1, 1, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 1), (1, 1, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 1), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 1), (0, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 0, 0, 1, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 1, 0, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 1, 1, 0), (1, 1, 1, 1, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 1, 1, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 1, 0), (1, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 0, 0, 1), (1, 0, 1, 1, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 1, 0), (1, 1, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 1, 0, 0, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0), (0, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 0), (0, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 0, 1, 1), (1, 0, 0, 1, 1, 0, 1, 0), (0, 0, 1, 0, 1, 0, 1, 0), (1, 0, 1, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 0, 0), (0, 1, 0, 0, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 1, 1, 0), (0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 1, 0, 1, 1), (1, 0, 1, 1, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (1, 0, 0, 1, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 1, 1, 1, 0, 0), (1, 1, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (0, 1, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 0), (0, 0, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0), (1, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 1), (0, 0, 1, 0, 0, 1, 1, 1), (1, 1, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 1, 0, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 1, 0, 0, 0), (1, 1, 0, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 1), (0, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 1, 0, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 0, 1, 1), (1, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 0, 1, 0, 0), (1, 1, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0), (0, 1, 0, 0, 1, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 0, 0, 1, 0, 1, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0), (0, 1, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1), (0, 0, 1, 1, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 0, 0, 0, 0, 1, 1), (0, 1, 0, 1, 0, 0, 0, 0), (0, 1, 1, 0, 1, 0, 0, 1), (0, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0), (0, 1, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 1, 1, 1), (0, 1, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 1), (0, 0, 1, 0, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0), (1, 1, 0, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 1, 0, 0), (0, 1, 0, 0, 1, 0, 1, 0), (0, 1, 0, 0, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 0, 1, 0, 1, 1), (1, 0, 1, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (0, 1, 0, 0, 1, 0, 1, 0), (0, 0, 0, 0, 1, 1, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 1, 1, 0, 1, 0), (1, 0, 1, 0, 0, 1, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1, 1, 1), (0, 1, 1, 1, 1, 0, 1, 0), (0, 1, 0, 0, 1, 1, 0, 0), (1, 1, 1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 1, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0), (1, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 0, 1, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 0, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 0), (0, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 1), (1, 1, 1, 0, 0, 0, 0, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0), (0, 1, 1, 0, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (1, 0, 0, 0, 0, 1, 0, 0), (1, 1, 1, 0, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 0, 0), (1, 0, 1, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 1, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0), (0, 0, 1, 0, 0, 0, 1, 0), (1, 1, 1, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 1), (1, 0, 1, 1, 1, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 1, 0, 1, 1, 1), (0, 1, 0, 1, 1, 0, 1, 0), (0, 1, 1, 1, 1, 1, 0, 1), (0, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 1, 1, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0), (1, 1, 0, 1, 1, 0, 0, 0), (0, 0, 1, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0)]),matrix([(0, 1, 1, 0, 0, 0, 0, 0), (1, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 0, 0, 1, 1, 1), (0, 1, 0, 0, 0, 1, 1, 1), (0, 1, 1, 1, 1, 0, 1, 1), (0, 1, 1, 1, 1, 1, 0, 0), (0, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 1, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0), (0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0), (0, 1, 1, 0, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 0, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 0), (0, 1, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 0, 0), (1, 0, 0, 1, 1, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1), (0, 1, 1, 0, 0, 1, 1, 0), (1, 1, 0, 0, 0, 1, 0, 0), (1, 0, 0, 1, 1, 0, 0, 1), (0, 0, 1, 1, 0, 0, 0, 1), (0, 1, 1, 0, 0, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 0), (0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 0), (0, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (0, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 1, 1, 1), (0, 1, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 1), (0, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 1, 1, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1), (1, 1, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 1), (1, 0, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 0, 1, 0, 1, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 1, 1), (1, 0, 1, 1, 0, 1, 0, 0), (1, 0, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 0, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (0, 1, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 0), (1, 0, 1, 1, 1, 0, 0, 1), (0, 1, 0, 1, 0, 1, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 0, 0, 1), (0, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (0, 1, 0, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 1, 1, 1), (0, 1, 0, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 0, 0, 1, 0, 1, 0, 1), (1, 1, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 1, 0), (0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 1, 0, 0, 1, 0), (1, 1, 1, 1, 0, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 0), (0, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 1, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 0, 1, 1, 1, 0), (1, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 0, 0, 1), (0, 0, 1, 0, 0, 1, 1, 1), (0, 1, 1, 0, 1, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 0), (1, 0, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 0), (0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 0, 1, 0, 1), (0, 1, 0, 0, 1, 1, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 0, 1, 0, 0, 1), (0, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 1, 1, 1, 0), (0, 0, 1, 1, 0, 1, 0, 0), (0, 1, 0, 0, 0, 1, 1, 1), (0, 1, 0, 0, 1, 0, 0, 1), (1, 0, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 0, 0), (1, 0, 1, 0, 0, 0, 0, 1), (0, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 1, 0, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0), (0, 0, 1, 0, 1, 1, 0, 0), (1, 1, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 1, 0), (1, 0, 1, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 1, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 0, 1, 1, 1, 0), (1, 1, 1, 1, 0, 0, 1, 0), (1, 0, 0, 1, 0, 0, 1, 0), (1, 0, 1, 1, 1, 1, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0)]),matrix([(0, 1, 1, 0, 1, 1, 0, 0), (1, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 1, 0, 0, 1, 1), (0, 1, 1, 0, 1, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 1, 1, 0, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 0, 1, 1), (1, 1, 1, 0, 0, 0, 0, 1), (0, 1, 1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 1, 0, 1, 1, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0), (0, 0, 1, 1, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 1, 1, 1, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 1), (1, 0, 0, 1, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 1, 1, 1, 0), (1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 1, 1, 0, 0, 1), (1, 0, 1, 1, 0, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 1, 0, 1), (0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 0), (1, 1, 0, 0, 1, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 0, 1), (0, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 0, 1, 0), (1, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 1, 1, 0, 0, 0), (1, 0, 1, 0, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 0), (1, 0, 0, 1, 1, 0, 0, 1), (1, 0, 1, 0, 1, 0, 0, 1), (1, 0, 1, 1, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 0), (1, 0, 1, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 1, 1, 1), (1, 1, 0, 0, 0, 0, 0, 0), (1, 1, 1, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0), (1, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 0, 0, 1, 0, 1, 1, 1), (0, 0, 1, 0, 0, 1, 1, 1), (1, 1, 0, 0, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 1), (0, 0, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 0, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 0, 0, 1, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 0, 0, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 0, 0, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (1, 1, 0, 1, 0, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 0, 0, 0, 0, 1, 1, 0), (1, 1, 1, 0, 0, 0, 1, 1), (1, 0, 0, 1, 0, 0, 1, 0), (1, 0, 1, 1, 1, 1, 0, 0), (0, 1, 1, 0, 1, 0, 0, 0)]),matrix([(0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 0, 1, 0, 0, 1, 0), (0, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 1, 1), (1, 0, 1, 0, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 0, 0, 0, 0), (1, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 0, 1, 0, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 0), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 1, 1), (0, 1, 1, 0, 0, 0, 1, 1), (1, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 1, 1, 1, 1, 0), (1, 0, 0, 0, 0, 1, 1, 1), (0, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 0, 1, 0), (1, 1, 1, 0, 0, 0, 0, 1), (0, 1, 1, 1, 1, 0, 0, 0), (1, 0, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 1), (1, 0, 0, 0, 1, 0, 0, 1), (0, 0, 0, 1, 1, 1, 1, 1), (0, 0, 1, 0, 0, 1, 1, 0), (1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 0, 1, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 1, 0), (1, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 1), (0, 1, 1, 0, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (1, 0, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 1, 0), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (1, 1, 0, 0, 0, 1, 1, 0), (0, 1, 1, 0, 0, 0, 1, 1), (1, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 1, 1, 0, 0, 0), (0, 1, 1, 0, 1, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0), (0, 0, 0, 1, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 0, 1, 1, 0, 1), (1, 0, 1, 1, 0, 1, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0), (1, 1, 1, 0, 0, 1, 0, 1), (0, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 0, 1, 0, 1), (1, 1, 1, 0, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 0, 1, 0, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1), (0, 0, 0, 1, 1, 1, 1, 0), (1, 0, 0, 0, 0, 1, 1, 1), (0, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 1, 1, 1), (1, 1, 1, 0, 1, 0, 1, 1), (0, 1, 1, 1, 1, 1, 0, 1), (1, 0, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 1, 0), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 0), (1, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 1, 1, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0), (0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 1), (0, 1, 0, 1, 1, 1, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (0, 1, 0, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 1, 1, 1), (0, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 1), (0, 1, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 1), (0, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 0, 0, 1), (1, 1, 1, 0, 1, 1, 0, 0), (0, 1, 1, 1, 0, 1, 1, 0), (1, 0, 0, 1, 1, 0, 1, 0), (0, 1, 0, 0, 1, 1, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0)]),matrix([(0, 0, 0, 1, 1, 1, 0, 1), (0, 0, 1, 0, 1, 1, 1, 1), (0, 1, 0, 1, 0, 1, 1, 0), (1, 0, 1, 0, 0, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 1, 0, 0, 0), (0, 1, 1, 1, 1, 0, 0, 1), (1, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 0), (0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 1, 1), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 1, 0, 0, 0), (0, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 1, 1, 0, 1, 1, 1, 1), (1, 0, 0, 0, 1, 1, 1, 1), (1, 0, 0, 1, 1, 1, 1, 1), (0, 0, 1, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 0, 0, 1), (1, 1, 1, 1, 0, 0, 1, 1), (1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 1, 1, 1, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 1, 1, 1), (0, 0, 1, 0, 0, 1, 1, 1), (1, 1, 0, 1, 1, 1, 1, 0), (1, 0, 1, 0, 1, 0, 1, 1), (1, 0, 1, 1, 0, 1, 1, 1), (1, 1, 1, 0, 1, 0, 1, 0), (1, 1, 1, 1, 1, 1, 0, 0), (1, 1, 0, 1, 1, 0, 0, 0)]),matrix([(0, 1, 0, 1, 1, 1, 1, 1), (1, 0, 1, 0, 1, 1, 1, 1), (0, 1, 0, 1, 1, 1, 1, 1), (1, 0, 1, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0), (1, 1, 1, 1, 0, 1, 0, 1), (1, 1, 1, 1, 1, 0, 1, 0)]),matrix([(0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0), (0, 1, 1, 1, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 1), (1, 1, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 1, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1, 1, 1), (1, 1, 0, 1, 1, 1, 1, 1), (1, 1, 1, 0, 1, 1, 1, 1), (1, 1, 1, 1, 0, 1, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1, 0, 1), (1, 1, 1, 1, 1, 1, 1, 0)])]]

# 10
switching_subgraphs["AH10"] = [Graph(A) for A in [matrix([(0, 0, 0, 0, 0, 0, 0, 1, 0, 0), (0, 0, 0, 0, 1, 1, 0, 1, 0, 0), (0, 0, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 1, 1, 1, 1, 1, 1, 0, 1, 1), (1, 0, 1, 1, 0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 1, 1, 1, 1, 1, 0), (1, 1, 1, 0, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 0, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 0, 0, 0, 0, 0, 0, 1), (1, 1, 0, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 0, 1, 1, 0, 0), (0, 1, 0, 0, 0, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 0, 1, 1, 1), (0, 0, 1, 0, 1, 1, 1, 0, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 1, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 0, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0, 1, 1), (1, 0, 1, 1, 1, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 0, 0, 1, 1, 0, 0), (0, 1, 1, 1, 0, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1, 0, 0, 0, 0), (0, 0, 1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 0, 0), (0, 0, 1, 1, 1, 1, 0, 1, 0, 0), (1, 1, 0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 0, 0, 0, 0, 0, 0), (0, 1, 1, 1, 0, 0, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0, 1, 1), (1, 1, 0, 1, 0, 0, 0, 0, 1, 1), (0, 0, 0, 0, 0, 1, 1, 1, 0, 0), (0, 0, 1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 1), (1, 1, 1, 1, 1, 0, 1, 1, 0, 1), (1, 1, 0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 1, 1), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 0, 1, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 1, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 1, 0, 0), (1, 1, 1, 0, 1, 1, 0, 1, 0, 0), (0, 0, 1, 0, 1, 1, 1, 0, 0, 0), (1, 1, 1, 1, 1, 0, 0, 0, 0, 1), (1, 1, 0, 0, 1, 0, 0, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 1, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 0, 0, 1, 1, 0, 0), (0, 1, 1, 1, 0, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 0, 0, 0, 0), (1, 1, 0, 1, 1, 1, 0, 0, 0, 0), (1, 1, 0, 0, 0, 1, 0, 0, 0, 0), (1, 1, 1, 1, 0, 1, 0, 0, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 1), (0, 0, 1, 1, 1, 0, 1, 1, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1, 1, 0)]),matrix([(0, 0, 1, 1, 0, 0, 0, 1, 1, 1), (0, 0, 1, 1, 1, 1, 0, 1, 1, 1), (1, 1, 0, 0, 1, 1, 0, 0, 0, 1), (1, 1, 0, 0, 1, 1, 1, 1, 0, 1), (0, 1, 1, 1, 0, 0, 1, 1, 0, 0), (0, 1, 1, 1, 0, 0, 1, 1, 1, 1), (0, 0, 0, 1, 1, 1, 0, 0, 1, 1), (1, 1, 0, 1, 1, 1, 0, 0, 1, 1), (1, 1, 0, 0, 0, 1, 1, 1, 0, 0), (1, 1, 1, 1, 0, 1, 1, 1, 0, 0)]),matrix([(0, 1, 0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 0, 0, 1, 1, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 1, 1, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 0, 0, 1, 0)]),matrix([(0, 0, 1, 0, 0, 0, 0, 1, 1, 0), (0, 0, 0, 1, 1, 1, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 1, 1, 1, 1, 0, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 1, 1, 1, 0), (1, 0, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 1, 1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1, 1, 0), (0, 1, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 0, 1, 0, 0, 1), (1, 0, 1, 0, 1, 1, 1, 0, 1, 0), (0, 1, 0, 1, 0, 1, 1, 1, 1, 0), (1, 0, 1, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1, 1, 1), (0, 1, 1, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 1, 0, 1, 0), (0, 0, 1, 0, 1, 1, 1, 0, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 0, 1, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 0, 0, 0, 1, 0, 0), (0, 1, 0, 1, 0, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 0, 1, 0), (1, 1, 0, 1, 1, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 1, 1, 1, 0), (0, 1, 1, 0, 0, 0, 1, 0, 0, 0), (0, 1, 0, 1, 0, 0, 0, 1, 1, 1), (0, 0, 0, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 0, 1, 1, 1, 0, 1, 0), (0, 0, 0, 1, 0, 0, 1, 0, 0, 1), (1, 0, 0, 0, 1, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 1, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 1, 1, 1), (1, 1, 0, 1, 1, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 0, 0, 0, 1, 1, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 1, 0, 0)]),matrix([(0, 1, 1, 0, 1, 1, 0, 1, 0, 1), (1, 0, 0, 1, 0, 0, 0, 1, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1, 1, 0), (0, 1, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1, 1, 0), (0, 1, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 0, 1, 1), (1, 0, 1, 0, 1, 0, 0, 1, 0, 0), (1, 1, 1, 0, 1, 0, 0, 1, 0, 1), (0, 0, 1, 0, 0, 1, 1, 0, 1, 0), (0, 1, 1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 1, 0, 1, 1, 1), (1, 0, 1, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 0, 0, 1, 0, 0, 1, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 1, 0, 0, 0, 1, 0, 1), (1, 0, 1, 0, 1, 1, 0, 1, 1, 0), (0, 1, 0, 1, 0, 1, 1, 1, 1, 0), (1, 0, 1, 0, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 1, 0, 1, 1, 1), (0, 1, 1, 0, 1, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 1, 0, 1, 0, 1), (1, 1, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 1, 1, 1, 0, 0, 1, 0, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 0)]),matrix([(0, 0, 0, 1, 0, 0, 0, 1, 1, 0), (0, 0, 1, 0, 1, 1, 0, 1, 0, 1), (0, 1, 0, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 1, 1, 1, 1, 0), (0, 1, 1, 0, 0, 0, 1, 0, 1, 1), (0, 1, 0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 0, 1, 1, 0, 0, 0, 1, 0), (1, 1, 0, 1, 0, 1, 0, 0, 0, 1), (1, 0, 1, 1, 1, 0, 1, 0, 0, 0), (0, 1, 0, 0, 1, 0, 0, 1, 0, 0)]),matrix([(0, 1, 1, 0, 1, 1, 1, 0, 0, 1), (1, 0, 0, 1, 0, 0, 1, 0, 1, 0), (1, 0, 0, 1, 0, 1, 1, 1, 0, 1), (0, 1, 1, 0, 1, 0, 0, 0, 0, 1), (1, 0, 0, 1, 0, 1, 0, 1, 0, 0), (1, 0, 1, 0, 1, 0, 1, 0, 1, 1), (1, 1, 1, 0, 0, 1, 0, 1, 0, 1), (0, 0, 1, 0, 1, 0, 1, 0, 1, 0), (0, 1, 0, 0, 0, 1, 0, 1, 0, 1), (1, 0, 1, 1, 0, 1, 1, 0, 1, 0)]),matrix([(0, 0, 0, 0, 0, 1, 0, 0, 0, 1), (0, 0, 1, 1, 0, 1, 1, 1, 0, 1), (0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 1, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 1, 1, 1, 0, 1, 1, 1, 0), (1, 0, 0, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 1, 1, 1, 1, 0, 1, 1), (1, 0, 1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 1, 1, 0, 1, 1, 1), (0, 0, 0, 1, 0, 0, 0, 0, 0, 1), (1, 1, 0, 1, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 1, 0, 1, 1, 1, 0), (1, 0, 1, 1, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 1, 1, 1, 0, 1, 1), (0, 1, 1, 0, 0, 0, 1, 0, 0, 0), (1, 1, 1, 0, 0, 1, 1, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 1, 1, 0, 1, 1, 1), (1, 1, 0, 1, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 0, 1, 0, 0, 0, 1), (0, 0, 0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1, 1, 1), (0, 0, 1, 0, 0, 0, 0, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 1, 1, 1, 0, 1, 1), (0, 1, 1, 0, 0, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 1, 1, 1, 1, 0), (1, 1, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 1, 0, 1, 1, 1, 0), (1, 0, 1, 1, 1, 0, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 1, 0, 1, 1), (0, 1, 1, 0, 1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 0, 1, 1, 1, 1, 0), (0, 0, 0, 1, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1, 1, 1), (1, 1, 1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 1, 1, 1, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 1, 1, 1, 0, 1, 1), (1, 1, 0, 1, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 1, 0, 1), (0, 1, 1, 1, 0, 1, 0, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 1, 0, 1, 1), (0, 1, 1, 0, 1, 1, 1, 0, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1, 1, 0), (1, 1, 0, 1, 1, 0, 0, 0, 1, 0), (1, 0, 1, 1, 1, 0, 0, 1, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 1, 1, 0, 1, 1, 1), (0, 1, 1, 0, 0, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 1, 1, 1, 1, 0), (1, 1, 1, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0, 1, 1, 0, 1), (1, 0, 0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 1, 1, 0, 1, 0, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0, 1, 1), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (1, 1, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1, 1, 0), (1, 0, 1, 1, 0, 1, 0, 0, 1, 0), (0, 1, 0, 1, 0, 0, 0, 1, 1, 1), (0, 1, 1, 0, 1, 1, 0, 1, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1, 1, 0), (1, 1, 0, 1, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 0, 0, 0), (1, 1, 1, 0, 1, 1, 1, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 0, 1, 0)]),matrix([(0, 0, 1, 1, 1, 0, 0, 0, 1, 0), (0, 0, 0, 0, 1, 0, 1, 1, 1, 0), (1, 0, 0, 0, 1, 1, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 1, 1), (1, 1, 1, 0, 0, 0, 0, 0, 0, 1), (0, 0, 1, 0, 0, 0, 1, 1, 0, 1), (0, 1, 1, 1, 0, 1, 0, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0, 1, 1), (1, 1, 0, 1, 0, 0, 0, 1, 0, 0), (0, 0, 0, 1, 1, 1, 0, 1, 0, 0)]),matrix([(0, 1, 0, 0, 0, 1, 1, 1, 0, 1), (1, 0, 1, 1, 0, 1, 0, 0, 0, 1), (0, 1, 0, 1, 0, 0, 0, 1, 1, 1), (0, 1, 1, 0, 1, 1, 0, 1, 0, 0), (0, 0, 0, 1, 0, 1, 1, 1, 1, 0), (1, 1, 0, 1, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 1, 0, 0, 1, 1, 1), (1, 0, 1, 1, 1, 0, 1, 0, 0, 0), (0, 0, 1, 0, 1, 1, 1, 0, 0, 1), (1, 1, 1, 0, 0, 0, 1, 0, 1, 0)])]]

## Switching

In [6]:
def switch(G: Graph, Q: matrix, C: list):
    A = G.adjacency_matrix(vertices=C+[x for x in G if not x in C])
    QI = block_diagonal_matrix([Q,identity_matrix(len(G)-Q.nrows())])
    Anew = QI.T*A*QI
    # check if the switching works
    if all(all(i==0 or i==1 for i in row) for row in Anew.rows()):
        return Graph(Anew).canonical_label()
    else:
        return False

## Finding all cospectral mates

In [7]:
def allCospectralMates(G, method, specific_subgraph=True):
    representatives = [G.canonical_label()]
    Q = switching_matrix[method]
    if specific_subgraph and method in switching_subgraphs.keys():
        # specific switching subgraph
        switchingsets = (C for Gamma in switching_subgraphs[method] for C in allSwitchingSets(G, Q, switching_subgraph=Gamma))
    else:
        # not checking for a specific switching subgraph
        switchingsets = allSwitchingSets(G, Q)
    for C in switchingsets:
        H = switch(G, Q, C)
        if H and not H in representatives:
            representatives.append(H)
            yield H

## Examples

In [8]:
list(allCospectralMates(graphs.PetersenGraph(), "GM4"))

[]

In [9]:
list(allCospectralMates(graphs.GrassmannGraph(2,4,2), "GM4"))

[Graph on 35 vertices, Graph on 35 vertices]